In [ ]:
import argparse
import os
import random

import numpy as np
import torch
import torchvision.models as models
from torchvision import transforms

from attack_utils import run_experiment
from impl_apgd_ce import APGD_CE_Linf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

datasets = {
    'correct_1000_AlexNet': ('alexnet', models.AlexNet_Weights.IMAGENET1K_V1),
    'correct_1000_DenseNet121': ('densenet121', models.DenseNet121_Weights.IMAGENET1K_V1),
    'correct_1000_GoogLeNet': ('googlenet', models.GoogLeNet_Weights.IMAGENET1K_V1),
    'correct_1000_MobileNetV3_large': ('mobilenet_v3_large', models.MobileNet_V3_Large_Weights.IMAGENET1K_V1),
    'correct_1000_ResNet34': ('resnet34', models.ResNet34_Weights.IMAGENET1K_V1),
    'correct_1000_VGG11': ('vgg11', models.VGG11_Weights.IMAGENET1K_V1),
    'correct_1000_EfficientNet_b0': ('efficientnet_b0', models.EfficientNet_B0_Weights.IMAGENET1K_V1),
}

config = {
    'SEED': 42,
    'selected_count': 500,
    'output_dir': 'adversarial_samples',
    'attack_name': 'apgd_ce',
    'MAX_SAVE_ADV': 10,
    'if_save_adv': False,
    'threshold': 1e-6,
    'target_labels': torch.tensor([100]).to(device),
    'if_target': False,
    'if_prune': False,
    'eps': 100/255,
    'alpha': 1/255,
    'steps': 100,
    'random_start': False,
    'targeted': False,
    'n_restarts': 2,
    'rho': 0.75,
    'eot_iter': 1,
    'early_stop': True,
    'step_ratio': 0.01,
    'max_ratio': 1.0,
}

parser = argparse.ArgumentParser(description='Adversarial Attack Experiment')
parser.add_argument('--eps', type=float, default=config['eps'], help='Epsilon for APGD-CE')
parser.add_argument('--alpha', type=float, default=config['alpha'], help='Initial step size for APGD-CE')
parser.add_argument('--steps', type=int, default=config['steps'], help='Steps for APGD-CE')
parser.add_argument('--n_restarts', type=int, default=config['n_restarts'], help='Restarts for APGD-CE')
parser.add_argument('--rho', type=float, default=config['rho'], help='Oscillation threshold for APGD-CE')
parser.add_argument('--eot_iter', type=int, default=config['eot_iter'], help='EOT iterations for APGD-CE')
args = parser.parse_args([])

config.update(vars(args))

SEED = config['SEED']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


使用设备: cuda


In [2]:
if __name__ == "__main__":
    for dataset_name, (model_name, weights) in datasets.items():
        print(f"\n=== 开始实验: {dataset_name} 使用模型 {model_name} ===")

        model = getattr(models, model_name)(weights=weights).to(device)
        model.eval()

        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.ToTensor(),
        ])

        config['val_dir'] = f"./{dataset_name}"
        config['attack_name'] = f"apgd_ce_{model_name}"

        val_dir = config['val_dir']
        all_images = [f for f in os.listdir(val_dir) if f.endswith('.JPEG')]
        selected_images = random.sample(all_images, min(config['selected_count'], len(all_images)))

        output_dir = config['output_dir']
        attack_name = config['attack_name']
        attack_output_dir = os.path.join(output_dir, attack_name)
        os.makedirs(attack_output_dir, exist_ok=True)

        results = run_experiment(
            config=config,
            transform=transform,
            model=model,
            device=device,
            selected_images=selected_images,
            attack_output_dir=attack_output_dir,
            attack_cls=APGD_CE_Linf,
        )

        print(f"=== 实验完成: {dataset_name} ===")



=== 开始实验: correct_1000_AlexNet 使用模型 alexnet ===


apgd_ce_alexnet: 100%|██████████| 500/500 [00:33<00:00, 14.88it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_ce_alexnet', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_AlexNet'}
成功攻击: 500/500
攻击成功率: 100.00%
平均被修改像素点数量: 85994.10
平均被修改像素比例: 99.14%
平均扰动均值: 14107.80
平均 SSIM: 0.913715
平均 PSNR: 36.1197dB
SSIM >= 0.975: 14.60%
SSIM >= 0.98: 9.60%
SSIM >= 0.985: 4.60%
SSIM >= 0.99: 1.80%
SSIM >= 0.995: 0.60%
PSNR >= 39dB: 15.00%
PSNR >= 41dB: 12.20%
PSNR >= 43dB: 0.80%
PSNR >= 45dB: 0.80%
PSNR >= 47dB: 0.80%
=== 实验完成: correct_1000_AlexNet ===

=== 开始实验: correct_1000_DenseNet121 使用模型 densenet121 ===


apgd_ce_densenet121: 100%|██████████| 500/500 [02:39<00:00,  3.13it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_ce_densenet121', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_DenseNet121'}
成功攻击: 500/500
攻击成功率: 100.00%
平均被修改像素点数量: 86071.36
平均被修改像素比例: 99.91%
平均扰动均值: 8982.76
平均 SSIM: 0.958026
平均 PSNR: 39.9356dB
SSIM >= 0.975: 34.40%
SSIM >= 0.98: 27.20%
SSIM >= 0.985: 18.60%
SSIM >= 0.99: 11.20%
SSIM >= 0.995: 1.60%
PSNR >= 39dB: 51.80%
PSNR >= 41dB: 51.60%
PSNR >= 43dB: 4.00%
PSNR >= 45dB: 3.80%
PSNR >= 47dB: 3.80%
=== 实验完成: correct_1000_DenseNet121 ===

=== 开始实验: correct_1000_GoogLeNet 使用模型 googlenet ===


...

配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_ce_resnet34', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_ResNet34'}
成功攻击: 500/500
攻击成功率: 100.00%
平均被修改像素点数量: 86561.37
平均被修改像素比例: 99.83%
平均扰动均值: 9163.96
平均 SSIM: 0.955738
平均 PSNR: 39.8925dB
SSIM >= 0.975: 35.60%
SSIM >= 0.98: 26.60%
SSIM >= 0.985: 17.20%
SSIM >= 0.99: 10.80%
SSIM >= 0.995: 4.20%
PSNR >= 39dB: 51.60%
PSNR >= 41dB: 51.60%
PSNR >= 43dB: 6.20%
PSNR >= 45dB: 5.80%
PSNR >= 47dB: 5.80%
=== 实验完成: correct_1000_ResNet34 ===

=== 开始实验: correct_1000_VGG11 使用模型 vgg11 ===


apgd_ce_vgg11: 100%|██████████| 500/500 [01:47<00:00,  4.63it/s]


配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_ce_vgg11', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_VGG11'}
成功攻击: 500/500
攻击成功率: 100.00%
平均被修改像素点数量: 86008.91
平均被修改像素比例: 99.86%
平均扰动均值: 10175.85
平均 SSIM: 0.945375
平均 PSNR: 39.1560dB
SSIM >= 0.975: 30.40%
SSIM >= 0.98: 22.00%
SSIM >= 0.985: 16.80%
SSIM >= 0.99: 10.00%
SSIM >= 0.995: 3.80%
PSNR >= 39dB: 46.40%
PSNR >= 41dB: 44.60%
PSNR >= 43dB: 5.40%
PSNR >= 45dB: 5.40%
PSNR >= 47dB: 5.40%
=== 实验完成: correct_1000_VGG11 ===

=== 开始实验: correct_1000_EfficientNet_b0 使用模型 efficientnet_b0 ===


apgd_ce_efficientnet_b0: 100%|██████████| 500/500 [01:52<00:00,  4.43it/s]

配置: {'SEED': 42, 'selected_count': 500, 'output_dir': 'adversarial_samples', 'attack_name': 'apgd_ce_efficientnet_b0', 'MAX_SAVE_ADV': 10, 'if_save_adv': False, 'threshold': 1e-06, 'target_labels': tensor([100], device='cuda:0'), 'if_target': True, 'if_prune': False, 'eps': 0.39215686274509803, 'alpha': 0.00392156862745098, 'steps': 100, 'random_start': False, 'targeted': True, 'n_restarts': 2, 'rho': 0.75, 'eot_iter': 1, 'early_stop': True, 'step_ratio': 0.01, 'max_ratio': 1.0, 'val_dir': './correct_1000_EfficientNet_b0'}
成功攻击: 500/500
攻击成功率: 100.00%
平均被修改像素点数量: 86573.62
平均被修改像素比例: 99.86%
平均扰动均值: 14776.63
平均 SSIM: 0.901729
平均 PSNR: 35.6004dB
SSIM >= 0.975: 12.20%
SSIM >= 0.98: 8.60%
SSIM >= 0.985: 4.60%
SSIM >= 0.99: 2.20%
SSIM >= 0.995: 0.60%
PSNR >= 39dB: 13.60%
PSNR >= 41dB: 10.20%
PSNR >= 43dB: 0.80%
PSNR >= 45dB: 0.60%
PSNR >= 47dB: 0.60%
=== 实验完成: correct_1000_EfficientNet_b0 ===
